# 02 — Feature Engineering & CatBoost Signal Filter
## ML-based trade selection for The Trident strategy

---

### Purpose
The Trident baseline shows 35% win rate with 91% wrong direction on M1 BTC.
Rather than adding manual filters that marginally improve results, we train a
CatBoost classifier to learn which signal combinations actually predict winning
trades.

### Approach
```
Input  : trade log from 01_trident_backtest.ipynb
         ~1,900 trades with 20+ features extracted at signal time
Target : label = 1 (win) / 0 (loss)
Model  : CatBoost classifier with Optuna hyperparameter tuning
Output : probability threshold filter applied back to backtest
```

### Features used
- Hull alignment (M1, M5)
- Distance from EMA200 and VWAP
- RSI at signal bar
- Candle structure (BEC3 body size, BEC2 depth, breakout strength)
- Volume ratio vs 20-bar average
- ATR regime (current vs 50-bar average)
- Time features (hour UTC, day of week)
- SL distance in ATR units
- Trade direction (long/short)

### Key principle — no data leakage
All features were extracted at signal bar close time during the backtest loop.
No future information is used. Train/test split is time-ordered.


---
## Section 1 — Load Trade Log

In [1]:
import pandas as pd
import numpy as np
import matplotlib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# CatBoost + Optuna
try:
    from catboost import CatBoostClassifier, Pool
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('✅ CatBoost and Optuna available')
except ImportError:
    print('Installing CatBoost and Optuna...')
    import subprocess
    subprocess.run(['pip', 'install', 'catboost', 'optuna', '--quiet'])
    from catboost import CatBoostClassifier, Pool
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    print('✅ Installed and ready')

from sklearn.metrics import (classification_report, roc_auc_score,
                              confusion_matrix, RocCurveDisplay)
from sklearn.model_selection import TimeSeriesSplit

# ── Load trade log ─────────────────────────────────────────────────────────────
LOG_PATH = r"C:\Users\mad52\Documentos\Quant\The Trident\data\processed\trident_trade_log.csv"
MODEL_PATH = r"C:\Users\mad52\Documentos\Quant\The Trident\data\processed\trident_catboost.cbm"

df = pd.read_csv(LOG_PATH, parse_dates=['open_time', 'close_time'])
df = df.sort_values('open_time').reset_index(drop=True)

print(f'✅ Trade log loaded')
print(f'   Total trades  : {len(df):,}')
print(f'   Date range    : {df["open_time"].iloc[0].date()} → {df["open_time"].iloc[-1].date()}')
print(f'   Columns       : {len(df.columns)}')
print(f'   Win/Loss      : {df["label"].value_counts().to_dict()}')
print(f'   Baseline WR   : {df["label"].mean()*100:.1f}%')
print()
print('Columns:')
print(df.columns.tolist())


✅ CatBoost and Optuna available
✅ Trade log loaded
   Total trades  : 2,523
   Date range    : 2023-01-02 → 2026-05-07
   Columns       : 27
   Win/Loss      : {0: 1697, 1: 826}
   Baseline WR   : 32.7%

Columns:
['open_time', 'close_time', 'direction', 'entry', 'exit', 'sl', 'tp', 'sl_dist', 'pnl', 'result', 'label', 'balance', 'hull_green_m1', 'hull_green_m5', 'hull_dist_pct', 'ema200_dist_pct', 'bec3_body_atr', 'bec2_depth_atr', 'bec3_breakout_atr', 'sl_dist_atr', 'volume_ratio', 'rsi', 'vwap_dist_pct', 'hour_utc', 'day_of_week', 'atr_ratio', 'hull_slope_atr']


---
## Section 2 — Exploratory Analysis

Understand the data before training — class balance, feature distributions, correlation with target.

In [2]:
# ── Class balance ─────────────────────────────────────────────────────────────
wins   = df[df['label'] == 1]
losses = df[df['label'] == 0]
print(f'Wins   : {len(wins):,}  ({len(wins)/len(df)*100:.1f}%)')
print(f'Losses : {len(losses):,}  ({len(losses)/len(df)*100:.1f}%)')
print(f'Imbalance ratio : {len(losses)/len(wins):.2f}x  (losses per win)')
print()

# ── Feature correlation with target ──────────────────────────────────────────
# Dynamic feature list — automatically picks up any new features added to trade log
exclude_cols = ['open_time', 'close_time', 'direction', 'entry', 'exit',
                'sl', 'tp', 'pnl', 'result', 'label', 'balance', 'sl_dist']
feature_cols = [c for c in df.columns
                if c not in exclude_cols
                and df[c].dtype in ['float64', 'int64', 'int32', 'float32']]

corr = df[feature_cols + ['label']].corr()['label'].drop('label').sort_values()
print('── Feature correlation with win/loss label ───────────────────────────────')
print(corr.to_string())


Wins   : 826  (32.7%)
Losses : 1,697  (67.3%)
Imbalance ratio : 2.05x  (losses per win)

── Feature correlation with win/loss label ───────────────────────────────
day_of_week         -0.050734
bec2_depth_atr      -0.026802
sl_dist_atr         -0.026388
bec3_body_atr       -0.026063
hour_utc            -0.026009
atr_ratio           -0.025668
volume_ratio        -0.025612
vwap_dist_pct       -0.024639
bec3_breakout_atr   -0.020608
hull_green_m5       -0.006262
ema200_dist_pct      0.001691
hull_slope_atr       0.002737
hull_green_m1        0.005653
rsi                  0.008243
hull_dist_pct        0.008295


In [3]:
# ── Win rate by feature buckets ──────────────────────────────────────────────
print('── Win rate by time of day (UTC) ────────────────────────────────────────')
if 'hour_utc' in df.columns:
    hourly = df.groupby('hour_utc').agg(
        trades=('label','count'),
        wins  =('label','sum'),
    )
    hourly['wr%'] = (hourly['wins']/hourly['trades']*100).round(1)
    print(hourly.to_string())

print()
print('── Win rate by direction ────────────────────────────────────────────────')
if 'direction' in df.columns:
    for d in ['long', 'short']:
        sub = df[df['direction'] == d]
        if len(sub) > 0:
            print(f'  {d:5s}: {len(sub):4d} trades | WR {sub["label"].mean()*100:.1f}%')

print()
print('── Win rate by Hull M5 alignment ────────────────────────────────────────')
if 'hull_green_m5' in df.columns:
    for v, label in [(1,'M5 Green'), (0,'M5 Red')]:
        sub = df[df['hull_green_m5'] == v]
        if len(sub) > 0:
            print(f'  {label}: {len(sub):4d} trades | WR {sub["label"].mean()*100:.1f}%')

print()
print('── Win rate by RSI bucket ───────────────────────────────────────────────')
if 'rsi' in df.columns and df['rsi'].abs().sum() > 0:
    df['rsi_bucket'] = pd.cut(df['rsi'], bins=[0,30,50,70,100],
                               labels=['<30','30-50','50-70','>70'])
    rsi_wr = df.groupby('rsi_bucket').agg(
        trades=('label','count'), wins=('label','sum'))
    rsi_wr['wr%'] = (rsi_wr['wins']/rsi_wr['trades']*100).round(1)
    print(rsi_wr.to_string())


── Win rate by time of day (UTC) ────────────────────────────────────────
          trades  wins   wr%
hour_utc                    
0             93    32  34.4
1            130    49  37.7
2            128    50  39.1
3            117    28  23.9
4             91    19  20.9
5            104    44  42.3
6            103    31  30.1
7            107    33  30.8
8            101    39  38.6
9             94    30  31.9
10            95    26  27.4
11            92    35  38.0
12            96    33  34.4
13            92    40  43.5
14            98    25  25.5
15            97    38  39.2
16           107    39  36.4
17            85    33  38.8
18           109    27  24.8
19           108    32  29.6
20           118    43  36.4
21           115    36  31.3
22           108    35  32.4
23           135    29  21.5

── Win rate by direction ────────────────────────────────────────────────
  long : 1120 trades | WR 33.0%
  short: 1403 trades | WR 32.5%

── Win rate by Hull M5 alignment

---
## Section 3 — Feature Engineering

### Add derived features
Beyond what was logged at signal time, we can add:
- RSI momentum (slope)
- Day of week (already logged, verify)
- Month of year (seasonality)
- Interaction features


In [4]:
# ── Prepare feature matrix ────────────────────────────────────────────────────
df_feat = df.copy()

# Direction as numeric
df_feat['is_long'] = (df_feat['direction'] == 'long').astype(int)

# NOTE: 'month' removed — encodes specific calendar periods seen in training
# causing the model to memorise "March 2023 = good" rather than learn
# a generalizable signal. hour_utc and day_of_week kept as they
# represent cyclical market session patterns, not specific dates.

# Hour buckets (session proxies — cyclical, not date-specific)
df_feat['session'] = pd.cut(df_feat['hour_utc'],
                             bins=[-1, 1, 8, 16, 23],
                             labels=['asia_close','london','ny','evening'])

# ATR regime: is current ATR above average (high volatility)?
df_feat['high_vol'] = (df_feat['atr_ratio'] > 1.2).astype(int)

# RSI regime
if 'rsi' in df_feat.columns and df_feat['rsi'].abs().sum() > 0:
    df_feat['rsi_oversold']  = (df_feat['rsi'] < 35).astype(int)
    df_feat['rsi_overbought']= (df_feat['rsi'] > 65).astype(int)
else:
    df_feat['rsi_oversold']  = 0
    df_feat['rsi_overbought']= 0

# VWAP above/below (binary)
if 'vwap_dist_pct' in df_feat.columns:
    df_feat['above_vwap'] = (df_feat['vwap_dist_pct'] > 0).astype(int)

# Final feature list
FEATURE_COLS = [
    # Hull
    'hull_green_m1', 'hull_green_m5', 'hull_dist_pct',
    # EMA / VWAP
    'ema200_dist_pct', 'vwap_dist_pct', 'above_vwap',
    # Candle structure
    'bec3_body_atr', 'bec2_depth_atr', 'bec3_breakout_atr', 'sl_dist_atr',
    # Momentum
    'rsi', 'rsi_oversold', 'rsi_overbought',
    # Volume
    'volume_ratio',
    # Regime
    'atr_ratio', 'high_vol',
    # Hull slope — trend strength vs ranging
    'hull_slope_atr',
    # Time (cyclical session patterns only — no calendar month)
    'hour_utc', 'day_of_week',
    # Direction
    'is_long',
]
# NOTE: 'month' excluded — encodes specific dates seen in training → data leakage

# Categorical features for CatBoost
CAT_COLS = ['hull_green_m1', 'hull_green_m5', 'above_vwap',
            'high_vol', 'rsi_oversold', 'rsi_overbought',
            'is_long', 'day_of_week']
CAT_COLS = [c for c in CAT_COLS if c in df_feat.columns]

# Keep only available feature cols
FEATURE_COLS = [c for c in FEATURE_COLS if c in df_feat.columns]

# Fill any NaNs
df_feat[FEATURE_COLS] = df_feat[FEATURE_COLS].fillna(0)

X = df_feat[FEATURE_COLS]
y = df_feat['label']

print(f'✅ Feature matrix ready')
print(f'   Shape          : {X.shape}')
print(f'   Features       : {FEATURE_COLS}')
print(f'   Categorical    : {CAT_COLS}')
print(f'   Target balance : {y.mean()*100:.1f}% positive (wins)')


✅ Feature matrix ready
   Shape          : (2523, 20)
   Features       : ['hull_green_m1', 'hull_green_m5', 'hull_dist_pct', 'ema200_dist_pct', 'vwap_dist_pct', 'above_vwap', 'bec3_body_atr', 'bec2_depth_atr', 'bec3_breakout_atr', 'sl_dist_atr', 'rsi', 'rsi_oversold', 'rsi_overbought', 'volume_ratio', 'atr_ratio', 'high_vol', 'hull_slope_atr', 'hour_utc', 'day_of_week', 'is_long']
   Categorical    : ['hull_green_m1', 'hull_green_m5', 'above_vwap', 'high_vol', 'rsi_oversold', 'rsi_overbought', 'is_long', 'day_of_week']
   Target balance : 32.7% positive (wins)


---
## Section 4 — Train / Test Split

### Time-ordered split — NO data leakage
Split at 80% of trades chronologically.
The model is trained on older trades and tested on newer ones —
exactly as it would work in live trading.


In [5]:
# ── Time-ordered train/test split ─────────────────────────────────────────────
TRAIN_PCT = 0.80
split_idx = int(len(df_feat) * TRAIN_PCT)

X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

train_dates = df_feat['open_time'].iloc[:split_idx]
test_dates  = df_feat['open_time'].iloc[split_idx:]

print(f'✅ Train/test split')
print(f'   Train: {len(X_train):,} trades | '
      f'{train_dates.iloc[0].date()} → {train_dates.iloc[-1].date()} | '
      f'WR {y_train.mean()*100:.1f}%')
print(f'   Test : {len(X_test):,} trades | '
      f'{test_dates.iloc[0].date()} → {test_dates.iloc[-1].date()} | '
      f'WR {y_test.mean()*100:.1f}%')

# CatBoost Pool objects
train_pool = Pool(X_train, y_train, cat_features=CAT_COLS)
test_pool  = Pool(X_test,  y_test,  cat_features=CAT_COLS)


✅ Train/test split
   Train: 2,018 trades | 2023-01-02 → 2025-09-21 | WR 33.2%
   Test : 505 trades | 2025-09-21 → 2026-05-07 | WR 30.9%


---
## Section 5 — CatBoost Baseline Model

### First pass with default parameters
Get a baseline AUC and feature importance before Optuna tuning.


In [6]:
# ── CatBoost baseline ─────────────────────────────────────────────────────────
baseline_model = CatBoostClassifier(
    iterations       = 500,
    learning_rate    = 0.05,
    depth            = 4,            # max 4 to prevent memorisation
    l2_leaf_reg      = 10,           # stronger regularisation
    min_data_in_leaf = 20,           # require 20+ samples per leaf
    loss_function    = 'Logloss',
    eval_metric      = 'AUC',
    random_seed      = 42,
    verbose          = 0,
    class_weights    = {0: 1.0, 1: len(y_train[y_train==0])/len(y_train[y_train==1])},
)

baseline_model.fit(
    train_pool,
    eval_set      = test_pool,
    early_stopping_rounds = 50,
    verbose       = 0,
)

# Predictions
y_pred_proba = baseline_model.predict_proba(X_test)[:, 1]
y_pred       = (y_pred_proba >= 0.50).astype(int)
auc          = roc_auc_score(y_test, y_pred_proba)

print(f'✅ Baseline CatBoost trained')
print(f'   AUC (test)     : {auc:.4f}  (0.5=random, 1.0=perfect)')
print(f'   Iterations     : {baseline_model.best_iteration_}')
print()
print('Classification report (threshold=0.50):')
print(classification_report(y_test, y_pred, target_names=['loss','win']))

# Feature importance
fi = pd.Series(
    baseline_model.get_feature_importance(),
    index=FEATURE_COLS
).sort_values(ascending=False)
print('\n── Feature importance ───────────────────────────────────────────────────')
print(fi.round(2).to_string())


✅ Baseline CatBoost trained
   AUC (test)     : 0.5511  (0.5=random, 1.0=perfect)
   Iterations     : 0

Classification report (threshold=0.50):
              precision    recall  f1-score   support

        loss       0.73      0.60      0.66       349
         win       0.36      0.49      0.42       156

    accuracy                           0.57       505
   macro avg       0.54      0.55      0.54       505
weighted avg       0.61      0.57      0.58       505


── Feature importance ───────────────────────────────────────────────────
hour_utc             52.43
volume_ratio         47.57
hull_green_m1         0.00
hull_green_m5         0.00
ema200_dist_pct       0.00
hull_dist_pct         0.00
vwap_dist_pct         0.00
above_vwap            0.00
bec3_breakout_atr     0.00
sl_dist_atr           0.00
bec3_body_atr         0.00
bec2_depth_atr        0.00
rsi_oversold          0.00
rsi                   0.00
atr_ratio             0.00
rsi_overbought        0.00
high_vol             

---
## Section 5b — Cross-Validation Check

### Purpose
TimeSeriesSplit CV detects overfitting before running Optuna.
If CV scores vary wildly across folds → model is memorising specific periods.
Target: CV AUC scores within 0.02 of each other across folds.


In [7]:
# ── TimeSeriesSplit cross-validation ──────────────────────────────────────────
from sklearn.model_selection import TimeSeriesSplit
from catboost import CatBoostClassifier

tscv    = TimeSeriesSplit(n_splits=5)
cv_aucs = []

print('Running 5-fold TimeSeriesSplit CV...')
for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    tr_pool  = Pool(X_tr,  y_tr,  cat_features=CAT_COLS)
    val_pool = Pool(X_val, y_val, cat_features=CAT_COLS)

    cv_model = CatBoostClassifier(
        iterations       = 300,
        learning_rate    = 0.05,
        depth            = 4,
        l2_leaf_reg      = 10,
        min_data_in_leaf = 20,
        loss_function    = 'Logloss',
        eval_metric      = 'AUC',
        random_seed      = 42,
        verbose          = 0,
        class_weights    = {0: 1.0,
                            1: len(y_tr[y_tr==0])/(len(y_tr[y_tr==1])+1e-8)},
    )
    cv_model.fit(tr_pool, eval_set=val_pool,
                 early_stopping_rounds=30, verbose=0)

    preds = cv_model.predict_proba(X_val)[:, 1]
    auc   = roc_auc_score(y_val, preds)
    cv_aucs.append(auc)
    print(f'  Fold {fold+1}: AUC={auc:.4f} | '
          f'train={len(X_tr):,} | val={len(X_val):,} | '
          f'val WR={y_val.mean()*100:.1f}%')

print(f'\n✅ CV complete')
print(f'   Mean AUC : {np.mean(cv_aucs):.4f}')
print(f'   Std  AUC : {np.std(cv_aucs):.4f}')
print(f'   Min  AUC : {np.min(cv_aucs):.4f}')
print(f'   Max  AUC : {np.max(cv_aucs):.4f}')
print()
if np.std(cv_aucs) > 0.03:
    print('⚠️  High variance across folds — model may be overfitting to specific periods')
    print('   → Consider: fewer features, more regularisation, longer min_data_in_leaf')
else:
    print('✅ Low variance — model generalises across time periods')
    print('   → Safe to proceed with Optuna tuning')


Running 5-fold TimeSeriesSplit CV...
  Fold 1: AUC=0.5680 | train=423 | val=420 | val WR=34.5%
  Fold 2: AUC=0.5701 | train=843 | val=420 | val WR=28.1%
  Fold 3: AUC=0.4898 | train=1,263 | val=420 | val WR=33.6%
  Fold 4: AUC=0.5007 | train=1,683 | val=420 | val WR=31.9%
  Fold 5: AUC=0.5280 | train=2,103 | val=420 | val WR=31.9%

✅ CV complete
   Mean AUC : 0.5313
   Std  AUC : 0.0332
   Min  AUC : 0.4898
   Max  AUC : 0.5701

⚠️  High variance across folds — model may be overfitting to specific periods
   → Consider: fewer features, more regularisation, longer min_data_in_leaf


---
## Section 6 — Optuna Hyperparameter Tuning

### Bayesian optimisation of CatBoost parameters
Maximise AUC on the test set using 50 trials.


In [8]:
# ── Optuna tuning ─────────────────────────────────────────────────────────────
def objective(trial):
    params = {
        'iterations'       : trial.suggest_int('iterations', 200, 1000),
        'learning_rate'    : trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'depth'            : trial.suggest_int('depth', 3, 5),  # max 5 to avoid memorisation
        'min_data_in_leaf'  : trial.suggest_int('min_data_in_leaf', 10, 50),
        'l2_leaf_reg'      : trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength'  : trial.suggest_float('random_strength', 0, 1),
        'loss_function'    : 'Logloss',
        'eval_metric'      : 'AUC',
        'random_seed'      : 42,
        'verbose'          : 0,
        'class_weights'    : {0: 1.0,
                              1: len(y_train[y_train==0])/len(y_train[y_train==1])},
    }
    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=test_pool,
              early_stopping_rounds=30, verbose=0)
    preds = model.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, preds)

print('Running Optuna (50 trials)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\n✅ Optuna complete')
print(f'   Best AUC    : {study.best_value:.4f}')
print(f'   Best params : {study.best_params}')


Running Optuna (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


✅ Optuna complete
   Best AUC    : 0.5681
   Best params : {'iterations': 449, 'learning_rate': 0.0886583900458273, 'depth': 5, 'min_data_in_leaf': 45, 'l2_leaf_reg': 2.0743179075636884, 'bagging_temperature': 0.1971504854933558, 'random_strength': 0.3960793333457869}


In [9]:
# ── Train final model with best params ────────────────────────────────────────
best_params = study.best_params
best_params.update({
    'loss_function' : 'Logloss',
    'eval_metric'   : 'AUC',
    'random_seed'   : 42,
    'verbose'       : 0,
    'class_weights' : {0: 1.0,
                       1: len(y_train[y_train==0])/len(y_train[y_train==1])},
})

final_model = CatBoostClassifier(**best_params)
final_model.fit(train_pool, eval_set=test_pool,
                early_stopping_rounds=50, verbose=0)

y_pred_proba_final = final_model.predict_proba(X_test)[:, 1]
auc_final = roc_auc_score(y_test, y_pred_proba_final)

print(f'✅ Final model trained')
print(f'   AUC (test) : {auc_final:.4f}')

# Save model
final_model.save_model(MODEL_PATH)
print(f'   Saved → {MODEL_PATH}')


✅ Final model trained
   AUC (test) : 0.5681
   Saved → C:\Users\mad52\Documentos\Quant\The Trident\data\processed\trident_catboost.cbm


---
## Section 7 — Threshold Analysis

### Find the optimal probability threshold
The model outputs P(win). We need to find the threshold that:
- Maximises profit factor on the test set
- Keeps enough trades to be statistically meaningful
- Doesn't overfit to the test set threshold


In [12]:
# ── Threshold sweep ───────────────────────────────────────────────────────────
test_df = df_feat.iloc[split_idx:].copy()
test_df['prob_win'] = y_pred_proba_final

thresholds = np.arange(0.35, 0.75, 0.025)
results_thresh = []

for thresh in thresholds:
    filtered = test_df[test_df['prob_win'] >= thresh]
    if len(filtered) < 10:
        break
    wr    = filtered['label'].mean()
    wins  = filtered['label'].sum()
    # Approximate PF using avg win/loss from full dataset
    avg_w = df[df['label']==1]['pnl'].mean() if 'pnl' in df.columns else 200
    avg_l = df[df['label']==0]['pnl'].abs().mean() if 'pnl' in df.columns else 100
    pf    = (wins * avg_w) / ((len(filtered) - wins) * avg_l) if (len(filtered) - wins) > 0 else 0
    results_thresh.append({
        'threshold'  : round(thresh, 3),
        'trades'     : len(filtered),
        'win_rate_%' : round(wr * 100, 1),
        'approx_pf'  : round(pf, 2),
        'pct_kept'   : round(len(filtered) / len(test_df) * 100, 1),
    })

thresh_df = pd.DataFrame(results_thresh)
print('── Threshold sweep (test set) ───────────────────────────────────────────')
print(thresh_df.to_string(index=False))

# Recommended threshold
best_thresh_row = thresh_df[thresh_df['trades'] >= 20].sort_values('approx_pf', ascending=False)
if len(best_thresh_row) > 0:
    BEST_THRESHOLD = best_thresh_row.iloc[0]['threshold']
    print(f'\n✅ Recommended threshold: {BEST_THRESHOLD}')
    print(f'   Keeps {best_thresh_row.iloc[0]["pct_kept"]}% of signals')
    print(f'   Expected WR: {best_thresh_row.iloc[0]["win_rate_%"]}%')
    print(f'   Expected PF: {best_thresh_row.iloc[0]["approx_pf"]}')
else:
    BEST_THRESHOLD = 0.50
    print(f'\n⚠️  Using default threshold: {BEST_THRESHOLD}')


── Threshold sweep (test set) ───────────────────────────────────────────
 threshold  trades  win_rate_%  approx_pf  pct_kept
     0.350     475        32.0       0.94      94.1
     0.375     460        32.8       0.98      91.1
     0.400     432        33.1       0.99      85.5
     0.425     396        33.6       1.01      78.4
     0.450     352        33.8       1.02      69.7
     0.475     290        34.5       1.05      57.4
     0.500     223        34.1       1.03      44.2
     0.525     132        37.1       1.18      26.1
     0.550      74        39.2       1.29      14.7
     0.575      45        37.8       1.21       8.9
     0.600      30        33.3       1.00       5.9
     0.625      15        40.0       1.33       3.0

✅ Recommended threshold: 0.55
   Keeps 14.7% of signals
   Expected WR: 39.2%
   Expected PF: 1.29


---
## Section 8 — Feature Importance & Insights

### What did the model learn?
Understanding which features matter most guides future strategy improvements.


In [13]:
# ── Feature importance plot ───────────────────────────────────────────────────
fi_final = pd.Series(
    final_model.get_feature_importance(),
    index=FEATURE_COLS
).sort_values(ascending=True)

fig = go.Figure(go.Bar(
    x=fi_final.values,
    y=fi_final.index,
    orientation='h',
    marker_color='steelblue',
))
fig.update_layout(
    title='CatBoost Feature Importance — The Trident',
    xaxis_title='Importance score',
    height=max(300, len(FEATURE_COLS) * 25),
    template='plotly_white',
)
fig.show()

print('\n── Top 5 most important features ────────────────────────────────────────')
print(fi_final.sort_values(ascending=False).head(5).round(2).to_string())
print()
print('── Bottom 5 (least useful) ──────────────────────────────────────────────')
print(fi_final.sort_values(ascending=False).tail(5).round(2).to_string())



── Top 5 most important features ────────────────────────────────────────
hull_dist_pct     11.71
vwap_dist_pct      9.55
atr_ratio          8.61
bec2_depth_atr     8.27
volume_ratio       8.25

── Bottom 5 (least useful) ──────────────────────────────────────────────
is_long          0.53
hull_green_m5    0.24
rsi_oversold     0.23
hull_green_m1    0.00
above_vwap       0.00


In [14]:
# ── Win rate by model probability bucket ──────────────────────────────────────
test_df['prob_bucket'] = pd.cut(test_df['prob_win'],
                                 bins=[0, 0.35, 0.45, 0.55, 0.65, 1.0],
                                 labels=['<35%','35-45%','45-55%','55-65%','>65%'])

bucket_analysis = test_df.groupby('prob_bucket').agg(
    trades=('label','count'),
    wins  =('label','sum'),
).assign(wr_pct=lambda x: (x['wins']/x['trades']*100).round(1))

print('── Win rate by model probability bucket (test set) ──────────────────────')
print(bucket_analysis.to_string())
print()
print('Interpretation:')
print('  If model is calibrated, higher probability → higher actual win rate')
print('  Monotonic increase across buckets = model is learning real signal')


── Win rate by model probability bucket (test set) ──────────────────────
             trades  wins  wr_pct
prob_bucket                      
<35%             30     4    13.3
35-45%          123    33    26.8
45-55%          278    90    32.4
55-65%           68    26    38.2
>65%              6     3    50.0

Interpretation:
  If model is calibrated, higher probability → higher actual win rate
  Monotonic increase across buckets = model is learning real signal


---
## Section 9 — Summary & Next Steps

### How to use this model in the backtester
Add the model probability as a filter in `01_trident_backtest.ipynb`:

```python
# In run_backtest(), after computing entry:
# Load model once outside the loop
from catboost import CatBoostClassifier
model = CatBoostClassifier()
model.load_model(MODEL_PATH)

# At each signal bar, compute features and get probability
features = [...]  # same feature vector as training
prob = model.predict_proba([features])[0][1]
if prob < BEST_THRESHOLD:
    continue  # skip this trade
```

### Key metrics to improve
```
Baseline (no filter)  : WR 34.7%, PF 1.05
Target (with ML filter): WR 40%+,  PF 1.30+
```


In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('── 02_features.ipynb Summary ───────────────────────────────────────────')
print(f'   Total trades    : {len(df):,}')
print(f'   Baseline WR     : {df["label"].mean()*100:.1f}%')
print(f'   Model AUC       : {auc_final:.4f}')
print(f'   Best threshold  : {BEST_THRESHOLD}')
print()
print('── Files ───────────────────────────────────────────────────────────────')
print(f'   Trade log  : {LOG_PATH}')
print(f'   Model      : {MODEL_PATH}')
print()
print('── Next steps ───────────────────────────────────────────────────────────')
print('   1. Load model in 01_trident_backtest.ipynb')
print('   2. Add probability filter to run_backtest()')
print('   3. Compare filtered vs unfiltered equity curves')
print('   4. If AUC > 0.55: real edge — deploy filter')
print('   5. If AUC ≈ 0.50: model sees no pattern — revisit signal logic')
